# 06 — Hashtag Co-occurrence Network

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("06_hashtag_cooccurrence_network", total_steps=7, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook builds the hashtag co-occurrence network. Hashtags are nodes; an edge exists when two hashtags appear in the same tweet. This captures discourse structure and bridge hashtags.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets = pd.read_csv(PROCESSED / "tweets_with_entities.csv", parse_dates=["createdAt"])
tweets = parse_list_columns(tweets, ["hashtags"])

edges = make_pair_edges(tweets["hashtags"], source_col="source", target_col="target")
# Filter weak edges for interpretability; keep full edge table separately.
edges.to_csv(PROCESSED / "hashtag_edges_full.csv", index=False)
edges_filtered = edges[edges["weight"] >= 2].copy()
edges_filtered.to_csv(PROCESSED / "hashtag_edges_filtered_weight2.csv", index=False)
print("Full edges:", len(edges), "Filtered edges:", len(edges_filtered))
edges_filtered.head(20)


In [ ]:
progress.step("Purpose")
G = graph_from_edges(edges_filtered, source="source", target="target", weight="weight", directed=False)
summary = network_summary(G, "hashtag_cooccurrence")
metrics = compute_network_metrics(G)
communities = detect_communities(G)
metrics["community"] = metrics["node"].map(communities)
summary.to_csv(TABLES / "06_hashtag_network_summary.csv", index=False)
metrics.to_csv(PROCESSED / "hashtag_node_metrics.csv", index=False)
nx.write_gexf(G, NETWORKS / "hashtag_cooccurrence_network.gexf")
display(summary)
display(metrics.head(20))


In [ ]:
progress.step("Purpose")
# Top central hashtags by PageRank and bridge position
fig = px.bar(metrics.sort_values("pagerank", ascending=True).tail(25), x="pagerank", y="node", orientation="h", title="Top hashtags by PageRank centrality")
fig.update_layout(height=760, yaxis_title="Hashtag")
save_plotly(fig, INTERACTIVE / "06_top_hashtags_by_pagerank.html", FIGURES / "06_top_hashtags_by_pagerank.png")
fig.show()

fig = px.bar(metrics.sort_values("betweenness_centrality", ascending=True).tail(25), x="betweenness_centrality", y="node", orientation="h", title="Top bridge hashtags by betweenness centrality")
fig.update_layout(height=760, yaxis_title="Hashtag")
save_plotly(fig, INTERACTIVE / "06_top_bridge_hashtags.html", FIGURES / "06_top_bridge_hashtags.png")
fig.show()


In [ ]:
progress.step("Purpose")
# Popularity vs structural importance
hashtag_freq = pd.read_csv(PROCESSED / "hashtag_frequency.csv") if (PROCESSED / "hashtag_frequency.csv").exists() else pd.DataFrame(columns=["hashtag", "count"])
plot_df = metrics.merge(hashtag_freq, left_on="node", right_on="hashtag", how="left")
fig = px.scatter(plot_df, x="count", y="pagerank", size="weighted_degree", color="community", hover_name="node", title="Hashtag popularity vs network centrality")
fig.update_layout(height=620, xaxis_title="Raw hashtag frequency", yaxis_title="PageRank")
save_plotly(fig, INTERACTIVE / "06_hashtag_frequency_vs_pagerank.html", FIGURES / "06_hashtag_frequency_vs_pagerank.png")
fig.show()


In [ ]:
progress.step("save_pyvis_network(G, INTERACTIVE / '06_hashtag_network_interactive.html', title='Hashtag co-occurrence network', node_m")
# Interactive network map
save_pyvis_network(G, INTERACTIVE / "06_hashtag_network_interactive.html", title="Hashtag co-occurrence network", node_metrics=metrics, max_nodes=700)
print("Saved interactive network:", INTERACTIVE / "06_hashtag_network_interactive.html")


In [ ]:
progress.step("community_summary = metrics.groupby('community', as_index=False).agg(")
# Community summary
community_summary = metrics.groupby("community", as_index=False).agg(
    node_count=("node", "count"),
    avg_pagerank=("pagerank", "mean"),
    total_weighted_degree=("weighted_degree", "sum")
).sort_values("node_count", ascending=False)
community_summary.to_csv(TABLES / "06_hashtag_community_summary.csv", index=False)
community_summary.head(20)


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
